# 01 — Inheritance and `super()`

Inheritance lets a class *reuse* and *extend* another class. Use it when one class
**is-a** version of another. (Be more skeptical of that than you think — see the
composition section in `04_shapes_app` later.)

## A base class and a subclass

In [1]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

class Dog(Animal):                      # Dog inherits from Animal
    def speak(self):                    # OVERRIDE — same name, new behavior
        return f"{self.name} says woof!"

fido = Dog("Fido")
print(fido.speak())                     # uses Dog.speak
print(fido.name)                        # inherited from Animal.__init__
print(isinstance(fido, Dog), isinstance(fido, Animal))    # True True

Fido says woof!
Fido
True True


### Attribute lookup walks the **MRO** (Method Resolution Order)

When you look up `fido.speak`, Python walks the chain: `Dog` → `Animal` → `object`.
First match wins.

In [2]:
print(Dog.__mro__)        # (Dog, Animal, object)
# everything inherits from `object` ultimately. That's where __repr__, __eq__, etc. live.

(<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


## `super()` — call the parent's version

When you override a method but still want the parent's behavior *plus* something extra, call `super().method(...)`. The most common place: `__init__`.

In [3]:
class Animal:
    def __init__(self, name):
        self.name = name
        print(f"Animal.__init__ ran for {name}")

    def speak(self):
        return f"{self.name} makes a sound"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)          # let Animal set self.name
        self.breed = breed              # add Dog-specific state
        print(f"Dog.__init__ ran for {breed}")

    def speak(self):
        base = super().speak()          # use the parent's sentence
        return base + " — specifically, a woof!"

fido = Dog("Fido", "corgi")
print(fido.speak())
print(fido.name, fido.breed)

Animal.__init__ ran for Fido
Dog.__init__ ran for corgi
Fido makes a sound — specifically, a woof!
Fido corgi


### Why `super()` and not `Animal.__init__(self, name)`?

Two reasons:

1. **You don't have to repeat the parent's name.** If the class hierarchy changes, you don't have to chase down every direct reference.
2. **It cooperates with multiple inheritance** (rare in everyday code, but it matters for libraries). `super()` follows the MRO; a hard-coded `Animal.__init__` does not.

Use `super()` always. Hard-coded parent calls are a smell.

## 3-level chain — watch the order

In [4]:
class A:
    def __init__(self):
        print("A.__init__ start")
        print("A.__init__ end")

class B(A):
    def __init__(self):
        print("B.__init__ start")
        super().__init__()
        print("B.__init__ end")

class C(B):
    def __init__(self):
        print("C.__init__ start")
        super().__init__()
        print("C.__init__ end")

C()
print("---")
print("MRO:", C.__mro__)

C.__init__ start
B.__init__ start
A.__init__ start
A.__init__ end
B.__init__ end
C.__init__ end
---
MRO: (<class '__main__.C'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>)


Read the output: control flows *into* the chain (C → B → A), then unwinds (A → B → C). This is the same nesting pattern as decorators stacking.

## Polymorphism — same interface, many implementations

When several classes share a method name, you can iterate over them and call that method without caring which type each one is. That's polymorphism — and it's what makes large systems composable.

In [5]:
class Animal:
    def __init__(self, name): self.name = name
    def speak(self): return f"{self.name} makes a sound"

class Dog(Animal):
    def speak(self): return f"{self.name} says woof"

class Cat(Animal):
    def speak(self): return f"{self.name} says meow"

class Cow(Animal):
    def speak(self): return f"{self.name} says moo"

animals = [Dog("Fido"), Cat("Mittens"), Cow("Bessie")]

# No isinstance, no elif chain — each object knows how to speak for itself.
for a in animals:
    print(a.speak())

Fido says woof
Mittens says meow
Bessie says moo


### Anti-pattern: `isinstance`-chains

This is what polymorphism replaces. If you find yourself writing this, ask whether the
behavior could live on the class instead:

```python
for a in animals:
    if isinstance(a, Dog):    print(f"{a.name} woofs")
    elif isinstance(a, Cat):  print(f"{a.name} meows")
    elif isinstance(a, Cow):  print(f"{a.name} moos")
```

Same output, much worse design — adding `Sheep` requires editing the loop everywhere, not just adding a class.

## Mini-exercise

1. Build an `Employee` base class with `name`, `salary`, and a `monthly_pay()` method. Add subclasses `HourlyEmployee` (extra: `hours_per_week`) and `Contractor` (extra: `flat_monthly`). Each overrides `monthly_pay()`. Loop over a mixed list and print everyone's pay.
2. In the `C → B → A` example above, what would happen if `B.__init__` *forgot* to call `super().__init__()`? Try it — predict first.
3. Refactor this `isinstance` chain into polymorphism:
   ```python
   def area(shape):
       if isinstance(shape, Circle):    return 3.14 * shape.r ** 2
       elif isinstance(shape, Square):  return shape.side ** 2
   ```

In [6]:
# your solutions here


**Takeaways**

- `class Sub(Base):` — `Sub` inherits everything from `Base`.
- Override by re-defining a method. Use `super().method(...)` to call the parent's version.
- Method lookup walks the MRO (`Cls.__mro__`).
- Polymorphism replaces `isinstance` chains — define behavior on the class, not in the caller.

Next: [02_abc_and_polymorphism.ipynb](02_abc_and_polymorphism.ipynb) — enforcing the interface.